In [1]:
#%pip install pyspark

In [2]:
#%pip install xarray

In [3]:
#%pip install netCDF4

In [4]:
#%pip install esgf-pyclient==0.3.1

In [ ]:
#%pip install dask

In [5]:
#%pip install pyarrow

In [2]:
from config.variables_config import map_variaveis_meta
from config.models_config import map_modelos_dir

In [2]:
map_variaveis_meta["tas"]

{'variable_id': 'tas',
 'variable_long_name': 'Near-Surface Air Temperature',
 'experiment_id': 'historical',
 'variant_label': 'r1i1p1f1'}

In [3]:
from pyspark.sql import SparkSession
import xarray as xr
import os
import sys
import glob
#os.environ["SPARK_LOCAL_DIR"] = "C:/tmp/spark"
#os.environ["SPARK_LOCAL_DIR"] = "C:/Users/thais/spark-temp"

## acesso ao ESGF

In [4]:
from collections import Counter, defaultdict
import logging
from pyesgf.search import SearchConnection
import requests
from copy import copy
import pytz
import datetime as dt
import matplotlib.pyplot as plt
import pandas as pd

selected_tz = pytz.timezone("America/Sao_Paulo")
os.environ["ESGF_PYCLIENT_NO_FACETS_STAR_WARNING"] = "1"

In [1]:
import hashlib

def compute_checksum(file_path, algorithm="sha256", chunk_size=8192):
    """
    calcula o checksum de um arquivo

    """
    
    h = hashlib.new(algorithm)
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

In [12]:
# arquivos
file_name = ["tas_Amon_EC-Earth3_historical_r1i1p1f1_gr_191001-191012.nc", "tas_Amon_EC-Earth3_historical_r1i1p1f1_gr_191101-191112.nc"]
year_dir = ["1910", "1911"]

list_nc_files = []

for f in range(len(file_name)):
    file_path = os.path.join(year_dir[f], file_name[f])

    # cálculo do checksum
    #checksum = compute_checksum(file_path)
    checksum = f"001{f}"

    list_nc_files.append({"path": file_path, "checksum": checksum})

In [13]:
list_nc_files

[{'path': '1910/tas_Amon_EC-Earth3_historical_r1i1p1f1_gr_191001-191012.nc',
  'checksum': '0010'},
 {'path': '1911/tas_Amon_EC-Earth3_historical_r1i1p1f1_gr_191101-191112.nc',
  'checksum': '0011'}]

In [5]:
# configurações de log
class TZFormatter(logging.Formatter):
    def __init__(self, fmt=None, datefmt=None, tz=None):
        super().__init__(fmt=fmt, datefmt=datefmt)
        self.tz = tz or pytz.UTC

    def formatTime(self, record, datefmt=None):
        date_time = dt.datetime.fromtimestamp(record.created, self.tz)
        if datefmt:
            s = date_time.strftime(datefmt)
        else:
            s = date_time.isoformat()
        return s

# função de configuração do log
def configurar_logger(nome_logger):
    logger = logging.getLogger(nome_logger)
    logger.setLevel(logging.INFO)
    logger.propagate = False  # evita envio ao root logger

    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)

        # Timezone Brasil (Horário de Brasília)
        #tz_brasil = pytz.timezone("America/Sao_Paulo")

        formatter = TZFormatter(
            fmt='%(asctime)s | %(name)s | %(levelname)s | %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S %z',
            tz=selected_tz
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger

logger_read = configurar_logger("leitura_dados")
logger_ingestion = configurar_logger("ingestao_dados")

In [10]:
# selecionando a variável
variavel_escolhida = map_variaveis_meta["tas"]

logger_read.info(f"variável escolhida: {variavel_escolhida['variable_id']}")
####################
# selecionando o modelo
modelo_escolhido = map_modelos_dir["EC-Earth3"]

logger_read.info(f"modelo escolhido: {modelo_escolhido['nome']}")

# conexão com ESGF e busca dos datasets
conn = SearchConnection("https://esgf-data.dkrz.de/esg-search", distrib=False)
try:
    logger_read.info(f"iniciando leitura da variável {variavel_escolhida['variable_id']} | modelo {modelo_escolhido['nome']} | {variavel_escolhida['experiment_id']}")

    ctx = conn.new_context(
        project="CMIP6",
        source_id=modelo_escolhido["nome"],#"EC-Earth3",
        experiment_id="ssp585",#variavel_escolhida["experiment_id"],#"historical",
        variable_id=variavel_escolhida["variable_id"],#"tas",
        table_id="Amon",
        frequency="mon",
        variant_label=variavel_escolhida["variant_label"]#"r110i1p1f1"
    )

    if ctx.hit_count:
        logger_read.info(f"dataset encontrado")

        # listagem das URLs dos arquivos do dataset tas | EC-Earth3 | historical | Amon | mon | r110i1p1f1
        result = ctx.search()[0]
        result.dataset_id
        files_url_tas = []

        files = result.file_context().search()
        for file in files:
            #print(file.opendap_url)
            files_url_tas.append(file)

        logger_read.info(f"quantidade de partições do dataset = {len(files_url_tas)}")
        #print(f"quantidade de partições do dataset = {len(files_url_tas)}")

        # lista reduzida de datasets
        qntd = 5
        logger_read.info(f"quantidade de partições que serão processadas: {qntd}")
        files_url_tas_shrunken = files_url_tas[:qntd]

    else:
        logger_read.info(f"dataset não encontrado")
except Exception as e:
    logger_read.error(f"erro de leitura da variável {variavel_escolhida['variable_id']} | modelo {modelo_escolhido['nome']} | {variavel_escolhida['experiment_id']}:\n{e}")

2025-08-23 23:12:56 -0300 | leitura_dados | INFO | variável escolhida: tas
2025-08-23 23:12:56 -0300 | leitura_dados | INFO | modelo escolhido: Earth3
2025-08-23 23:12:56 -0300 | leitura_dados | INFO | iniciando leitura da variável tas | modelo Earth3 | historical


2025-08-23 23:12:57 -0300 | leitura_dados | INFO | dataset não encontrado


In [5]:
# variáveis relevantes de acordo com GCOS
variaveis = ["tas", "tasmax", "tasmin", "ps", "pr", "hur", "rsds", "rlut", "uas", "vas", "tos", "sos", "sic"]
variaveis = ["tas", "tasmin", "tasmax", "tos", "ps", "pr"]  # exemplo com 6 variáveis

In [6]:
# diretórios camadas
dir_raiz = r"/home/thais/climate-ingestion/climate_data_processing/datasets"

dir_raw = os.path.join(dir_raiz, "raw")
dir_trusted = os.path.join(dir_raiz, "trusted")
dir_delivery = os.path.join(dir_raiz, "delivery")

os.makedirs(dir_raw, exist_ok=True)
os.makedirs(dir_trusted, exist_ok=True)
os.makedirs(dir_delivery, exist_ok=True)

In [7]:
# mapeamento dos modelos e diretórios
#[ "EC-Earth3", "CanESM5", "MPI-ESM1-2-LR", "IPSL-CM5A2-INCA"]
map_modelos_dir = {
    "CanESM5": "can_esm5",
    "EC-Earth3": "ec_earth3",
    "IPSL-CM5A2-INCA": "ipsl_cm5a_mr",
    "MPI-ESM1-2-LR": "mpi_esm1_2_lr"
}

### tas | UKESM1-0-LL | tutorial

In [6]:
conn = SearchConnection('https://esgf-data.dkrz.de/esg-search', distrib=True)
ctx = conn.new_context(
    project='CMIP6',
    source_id='UKESM1-0-LL',
    experiment_id='historical',
    variable='tas',
    frequency='mon',
    variant_label='r1i1p1f2',
    data_node='esgf-data3.ceda.ac.uk')
ctx.hit_count

0

In [14]:
import os
if 'HOME' not in os.environ:
    os.environ['HOME'] = os.environ.get('USERPROFILE', 'C:\\Users\\thais')

from pyesgf.logon import LogonManager

lm = LogonManager()
lm.logoff()
print("Logado?", lm.is_logged_on())

myproxy_host = 'esgf-data.dkrz.de'
lm.logon(username=None, password=None, hostname=myproxy_host)  # Isso abrirá um prompt interativo
print("Logado após autenticação?", lm.is_logged_on())

Logado? False
Enter myproxy username: 

TimeoutError: [WinError 10060] Uma tentativa de conexão falhou porque o componente conectado não respondeu
corretamente após um período de tempo ou a conexão estabelecida falhou
porque o host conectado não respondeu

### tas | EC-Earth3

In [ ]:
# variáveis relevantes de acordo com GCOS
variaveis = ["tas", "tasmax", "tasmin", "ps", "pr", "hur", "rsds", "rlut", "uas", "tos"]
variaveis = ["tas", "tasmin", "tasmax", "tos", "ps", "pr"]  # exemplo com 6 variáveis

In [65]:
map_variaveis_meta = {
    "tas": {
        "variable_long_name": "Near-Surface Air Temperature",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "tasmax": {
        "variable_long_name": "Near-Surface Maximum Air Temperature",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "tasmin": {
        "variable_long_name": "Near-Surface Minimum Air Temperature",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "ps": {
        "variable_long_name": "Surface Air Pressure",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "pr": {
        "variable_long_name": "Precipitation",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "hur": {
        "variable_long_name": "Relative Humidity",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "rsds": {
        "variable_long_name": "Downward Shortwave Radiation Flux at Surface",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "rlut": {
        "variable_long_name": "Upward Longwave Radiation Flux at Top of Atmosphere",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "uas": {
        "variable_long_name": "Eastward Wind Component at 10m",
        "experiment_id": "historical",
        "variant_label": "r1i1p1f1"
    },
    "tos": {
        "variable_long_name": "Sea Surface Temperature",
        "experiment_id": "historical",
        "variant_label": "r2i1p1f1"
    }
}

In [8]:
# pr
conn = SearchConnection("https://esgf-data.dkrz.de/esg-search", distrib=False)

ctx = conn.new_context(
    project="CMIP6",
    source_id="EC-Earth3",
    #experiment_id="ssp585",
    experiment_id="historical",
    variable_id="sic",
    table_id="Amon",
    frequency="mon"
    #,variant_label="r2i1p1f1"
    #,latest=True
)

print(f"datasets encontrados: {ctx.hit_count}")

if ctx.hit_count > 0:
    result = ctx.search()[0]
    print(result.json["variable_long_name"]) # variable_units
    print(result.json["variable_units"])

datasets encontrados: 0


In [59]:
for i in range(ctx.hit_count):
    result = ctx.search()[i]
    print(result.json["variable_long_name"])  # variable_units
    print(f'member_id = {result.json["member_id"]}')
    print(f'experiment_id = {result.json["experiment_id"]}')
    print(f'experiment_id = {result.json["experiment_title"]}')
    print(f'latest = {result.json["latest"]}')
    print("---------------------")

['Sea Surface Temperature']
member_id = ['r10i1p1f1']
experiment_id = ['historical']
experiment_id = ['all-forcing simulation of the recent past']
latest = True
---------------------
['Sea Surface Temperature']
member_id = ['r2i1p1f1']
experiment_id = ['historical']
experiment_id = ['all-forcing simulation of the recent past']
latest = True
---------------------
['Sea Surface Temperature']
member_id = ['r14i1p1f1']
experiment_id = ['historical']
experiment_id = ['all-forcing simulation of the recent past']
latest = True
---------------------
['Sea Surface Temperature']
member_id = ['r7i1p1f1']
experiment_id = ['historical']
experiment_id = ['all-forcing simulation of the recent past']
latest = True
---------------------


In [28]:
# diretório modelo
modelo_escolhido = "EC-Earth3"
dir_modelo = map_modelos_dir[modelo_escolhido]

# diretório varável
variavel_escolhida = "tas"

# variant label
variant_label_escolhida = "r1i1p1f1"

#### Bronze/Raw layer

Download dos datasets e armazenamento

In [9]:
# diretório para os datasets
#dir_ec_earth_tas = r"C:\Users\thais\Documents\repositorios\climate_data_processing\datasets\raw\ec_earth3\tas"
#dir_raw = dir_ec_earth_tas

dir_modelo_raw = os.path.join(dir_raw, dir_modelo, variavel_escolhida)
os.makedirs(dir_raw, exist_ok=True)

In [10]:
# inicializando o spark
spark = SparkSession.builder.appName("ClimateData").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/22 13:45:44 WARN Utils: Your hostname, VALFENDA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/22 13:45:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/22 13:45:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
# tas
conn = SearchConnection("https://esgf-data.dkrz.de/esg-search", distrib=False)
try:
    logger_read.info(f"iniciando leitura da variável tas | modelo EC-Earth3 | historical")

    ctx = conn.new_context(
        project="CMIP6",
        source_id=modelo_escolhido,#"EC-Earth3",
        #experiment_id="ssp585",
        experiment_id="historical",
        variable_id=variavel_escolhida,#"tas",
        table_id="Amon",
        frequency="mon",
        variant_label=variant_label_escolhida#"r110i1p1f1"
    )

    if ctx.hit_count:
        logger_read.info(f"dataset encontrado")

        # listagem das URLs dos arquivos do dataset tas | EC-Earth3 | historical | Amon | mon | r110i1p1f1
        result = ctx.search()[0]
        result.dataset_id
        files_url_tas = []

        files = result.file_context().search()
        for file in files:
            #print(file.opendap_url)
            files_url_tas.append(file)

        logger_read.info(f"quantidade de partições do dataset = {len(files_url_tas)}")
        #print(f"quantidade de partições do dataset = {len(files_url_tas)}")

        # lista reduzida de datasets
        qntd = 15
        logger_read.info(f"quantidade de partições que serão processadas: {qntd}")
        files_url_tas_shrunken = files_url_tas[:qntd]

    else:
        logger_read.info(f"dataset não encontrado")
except Exception as e:
    logger_read.error(f"erro de leitura da variável tas | modelo EC-Earth3 | historical")

2025-08-23 23:10:48 -0300 | leitura_dados | INFO | iniciando leitura da variável tas | modelo EC-Earth3 | historical
2025-08-23 23:10:48 -0300 | leitura_dados | ERROR | erro de leitura da variável tas | modelo EC-Earth3 | historical


In [14]:
# ingestão dos arquivos

list_nc_files = []

for dataset_tas in files_url_tas_shrunken:

    file_name = dataset_tas.json["title"]
    url = dataset_tas.download_url

    print(f"-----")
    logger_read.info(f"arquivo atual = {file_name}")
    
    try:
        response = requests.get(url)

        # extraindo o ano do nome do arquivo
        #year = file_name.split("_")[1]
        year_months = file_name.split("_")[-1].replace(".nc", "")
        year = year_months[:4]

        # criando diretório para o ano
        year_dir = os.path.join(dir_modelo_raw, year)
        os.makedirs(year_dir, exist_ok=True)

        # diretório completo do arquivo
        file_path = os.path.join(year_dir, file_name)

        # salvando o arquivo
        with open(file_path, "wb") as f:
            f.write(response.content)

        # verificando se o arquivo foi salvo corretamente
        if os.path.exists(file_path):
            list_nc_files.append(file_path)
            logger_read.info(f"arquivo salvo em {file_path}")

    except Exception as e:
        logger_read.error(f"erro ao baixar {file_name}: {e}")

#logger_read.info(f"\ntotal arquivos Bronze/Raw salvos: {len(list_nc_files)}")
logger_read.info(f"\n-----\nquantidade de partições salvas: {len(list_nc_files)}")

-----
2025-08-22 14:53:10 -0300 | leitura_dados | INFO | arquivo atual = tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197001-197012.nc
2025-08-22 14:53:14 -0300 | leitura_dados | INFO | arquivo salvo em /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1970/tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197001-197012.nc
-----
2025-08-22 14:53:14 -0300 | leitura_dados | INFO | arquivo atual = tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197101-197112.nc
2025-08-22 14:53:17 -0300 | leitura_dados | INFO | arquivo salvo em /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1971/tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197101-197112.nc
-----
2025-08-22 14:53:17 -0300 | leitura_dados | INFO | arquivo atual = tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197201-197212.nc
2025-08-22 14:53:20 -0300 | leitura_dados | INFO | arquivo salvo em /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1972/tas_Amon_EC-Ea

#### Silver/Trusted layer

ingestão para processamento local

In [19]:
# diretório trusted
#dir_trusted = r"C:\Users\thais\Documents\repositorios\climate_data_processing\datasets\trusted\ec_earth3\tas"

dir_modelo_trusted = os.path.join(dir_trusted, dir_modelo, variavel_escolhida)
os.makedirs(dir_modelo_trusted, exist_ok=True)

In [ ]:
# lista com os datasets salvos na camada raw
datasets_salvos_raw = glob.glob(os.path.join(dir_modelo_raw, "**", "*.nc"), recursive=True)

In [24]:
# destino
#output_path = os.path.join(dir_trusted, "parquet")

for nc_file in datasets_salvos_raw:
    logger_read.info(f"processando {nc_file}")

    # carregando o dataset
    ds = xr.open_dataset(nc_file)

    # convertendo de Kelvin para Celsius
    tas_celsius0 = copy(ds[variavel_escolhida] - 273.15)

    # ajustando longitude (0–360 -> -180–180)
    tas_celsius1 = copy(tas_celsius0.assign_coords(lon=(((ds.lon + 180) % 360) - 180)))
    tas_celsius2 = copy(tas_celsius1.sortby("lon"))

    # selecionando registros da América Latina
    ds_latam = copy(tas_celsius2.sel(lat=slice(-60, 15), lon=slice(-120, -30)))

    # dataframe temporário
    df_temp = ds_latam.to_dataframe().reset_index()
    df_temp["time"] = df_temp["time"].astype("datetime64[ms]")
    df_temp["year"] = df_temp["time"].dt.year
    df_temp["month"] = df_temp["time"].dt.month

    # spark dataframe
    df_spark = spark.createDataFrame(df_temp)

    # salvando na camada trusted
    (
        df_spark
        .write
        .mode("append")   # aqui não sobrescreve, vai acumulando
        .partitionBy("year", "month")
        .parquet(dir_modelo_trusted)
    )

    ds.close()

2025-08-22 15:04:49 -0300 | leitura_dados | INFO | processando /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1975/tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197501-197512.nc


25/08/22 15:05:04 WARN TaskSetManager: Stage 0 contains a task of very large size (1069 KiB). The maximum recommended task size is 1000 KiB.
25/08/22 15:05:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/08/22 15:05:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


2025-08-22 15:05:11 -0300 | leitura_dados | INFO | processando /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1971/tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197101-197112.nc


25/08/22 15:05:21 WARN TaskSetManager: Stage 1 contains a task of very large size (1049 KiB). The maximum recommended task size is 1000 KiB.


2025-08-22 15:05:23 -0300 | leitura_dados | INFO | processando /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1973/tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197301-197312.nc


...

25/08/22 15:07:25 WARN TaskSetManager: Stage 12 contains a task of very large size (1069 KiB). The maximum recommended task size is 1000 KiB.


2025-08-22 15:07:27 -0300 | leitura_dados | INFO | processando /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1983/tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_198301-198312.nc


25/08/22 15:07:37 WARN TaskSetManager: Stage 13 contains a task of very large size (1069 KiB). The maximum recommended task size is 1000 KiB.


2025-08-22 15:07:38 -0300 | leitura_dados | INFO | processando /home/thais/climate-ingestion/climate_data_processing/datasets/raw/ec_earth3/tas/1979/tas_Amon_EC-Earth3_historical_r110i1p1f1_gr_197901-197912.nc


25/08/22 15:07:49 WARN TaskSetManager: Stage 14 contains a task of very large size (1069 KiB). The maximum recommended task size is 1000 KiB.


In [5]:
file_tasmax = "tasmax_Amon_MPI-ESM1-2-LR_historical_r1i1p1f1_gn_191001-192912.nc"
nc_files = [file_tasmax]

# carregando o dataset
ds = xr.open_dataset(nc_files[0])

In [6]:
# original
ds = xr.open_mfdataset(nc_files,
                       engine="h5netcdf",
                       combine="by_coords",
                       parallel=True,
                       chunks={"time": 50})

In [8]:
# opção 1
ds = xr.open_mfdataset(nc_files,
                       engine="h5netcdf",
                       combine="nested",
                       concat_dim="time",
                       parallel=True,
                       chunks={"time": 50})

/tmp/ipykernel_43929/2336375111.py:2: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(nc_files,


In [10]:
if len(nc_files) == 1:
    print(f"abrindo um único dataset")
    ds = xr.open_dataset(nc_files[0], engine="h5netcdf", chunks={"time": 50})
else:
    print(f"abrindo múltiplos datasets")
    ds = xr.open_mfdataset(nc_files, engine="h5netcdf", combine="nested", concat_dim="time", parallel=True, chunks={"time": 50})


abrindo um único dataset


In [14]:
ds

<xarray.Dataset> Size: 18MB
Dimensions:    (time: 240, bnds: 2, lat: 96, lon: 192)
Coordinates:
  * time       (time) datetime64[ns] 2kB 1910-01-16T12:00:00 ... 1929-12-16T1...
  * lat        (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
  * lon        (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 354.4 356.2 358.1
    height     float64 8B ...
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) datetime64[ns] 4kB ...
    lat_bnds   (lat, bnds) float64 2kB ...
    lon_bnds   (lon, bnds) float64 3kB ...
    tasmax     (time, lat, lon) float32 18MB ...
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   0.0
    branch_time_in_parent:  0.0
    contact:                cmip6-mpi-esm@dkrz.de
    ...                     ...
    title:                  MPI-ESM1-2-LR output prepared for CMIP6
    variable_id:            tasmax
    variant_label:          r1i1p1f1
    license:                CMIP6 model data produced by MPI-M is licensed un...
    cmor_version:           3.5.0
    tracking_id:            hdl:21.14100/524d9d1e-1684-4d40-8135-326a665a7f06

In [15]:
anos = sorted(list(set(ds["time"].dt.year.values)))

In [1]:
anos

NameError: name 'anos' is not defined

In [18]:
# Coleta de metadados relevantes do dataset
metadata = {
    "source_id": ds.attrs.get("source_id"),
    "experiment_id": ds.attrs.get("experiment_id"),
    "variant_label": ds.attrs.get("variant_label"),
    "grid_label": ds.attrs.get("grid_label"),
    "nominal_resolution": ds.attrs.get("nominal_resolution"),
    "realm": ds.attrs.get("realm"),
    "tracking_id": ds.attrs.get("tracking_id"),
    "institution": ds.attrs.get("institution"),
    "creation_date": ds.attrs.get("creation_date"),
    "mip_era": ds.attrs.get("mip_era")
}

metadata


{'source_id': 'MPI-ESM1-2-LR',
 'experiment_id': 'historical',
 'variant_label': 'r1i1p1f1',
 'grid_label': 'gn',
 'nominal_resolution': '250 km',
 'realm': 'atmos',
 'tracking_id': 'hdl:21.14100/524d9d1e-1684-4d40-8135-326a665a7f06',
 'institution': 'Max Planck Institute for Meteorology, Hamburg 20146, Germany',
 'creation_date': '2019-09-04T13:21:26Z',
 'mip_era': 'CMIP6'}

In [20]:
start_year = pd.to_datetime(ds["time"].values[0]).year

In [21]:
start_year

1910

In [ ]:
dt.datetime.fromtimestamp(start_time).isoformat()

#### Gold/Delivery layer